## Combine all datasets and Train Test Spilt

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import numpy as np
sys.path.append('../../')   # Add parent directory to Python path
import pickle
from utils.preprocessing import *
from utils.segmentation import *
from utils.visualization import *

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split

np.random.seed(42)  # For reproducibility


## 1. Combine all datasets

In [4]:
# Load the curb data (which appears to be stored as a dictionary with scene_0 and scene_1)
with open('../data_1s_30hz_real_world/real_world_1s_30hz_2people.npz', 'rb') as f:
    curb_data = np.load(f)
    data_curb_0 = curb_data['segments'][curb_data['labels'] == 0]
    data_curb_1 = curb_data['segments'][curb_data['labels'] == 1]

# Print shapes to verify the data was loaded correctly
print("Curb (scene 0):", data_curb_0.shape)
print("Curb (scene 1):", data_curb_1.shape)


Curb (scene 0): (204, 30, 3)
Curb (scene 1): (204, 30, 3)


In [5]:
# Add labels
# Assign labels for each scene type
datasets = [
    (data_curb_0, "non_curb"),  # scene 0 is non-curb
    (data_curb_1, "curb_1")     # scene 1 is curb
]

# Combine all into a single list of (sensor_values, label)
combined_dataset = []
for data, label in datasets:
    for segment in data:
        combined_dataset.append((segment, label))

# Print dataset information
print("\nDataset Summary:")
print(f"Scene 0 (non-curb): {data_curb_0.shape}")
print(f"Scene 1 (curb): {data_curb_1.shape}")
print(f"Total combined samples: {len(combined_dataset)}")


Dataset Summary:
Scene 0 (non-curb): (204, 30, 3)
Scene 1 (curb): (204, 30, 3)
Total combined samples: 408


## 3. Normalise dataset

In [16]:
X_train_normalized = normalize_3d_data(X_train)

# 4. Labels from string to integer

In [17]:
# Define custom mapping: curb_1=1, non_curb=0
custom_mapping = {"curb_1": 1, "non_curb": 0}

# Apply the custom mapping
y_train_int = np.array([custom_mapping[label] for label in y_train])
y_test_int = np.array([custom_mapping[label] for label in y_test])

# Create and manually adjust the label_encoder to match your encoding
label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(["non_curb", "curb_1"])  # Ensures 0=non_curb, 1=curb_1

print("Classes:", label_encoder.classes_)
print("First 10 y_train_int:", y_train_int[:10])
print("First 10 y_test_int:", y_test_int[:10])
for idx, label in enumerate(label_encoder.classes_):
    print(f"{idx}: {label}")

Classes: ['non_curb' 'curb_1']
First 10 y_train_int: [1 1 1 1 0 1 0 1 0 1]
First 10 y_test_int: [1 1 0 0 0 0 0 1 1 1]
0: non_curb
1: curb_1


## 5: One-hot encode the labels

In [18]:
y_train_onehot = to_categorical(y_train_int)
y_test_onehot = to_categorical(y_test_int)

print(y_train_onehot.shape)
print(y_test_onehot.shape)

(441, 2)
(111, 2)


In [19]:
# Randomly select an index and check that the one-hot encoding matches the original label
r = np.random.randint(len(y_train_int))
assert y_train_onehot[r].argmax() == y_train_int[r]
r = np.random.randint(len(y_test_int))
assert y_test_onehot[r].argmax() == y_test_int[r]

## 6. Save train, test data and labels

In [20]:
# Save test data
with open('../data_1s_30hz_real_world/X_test_data.pkl', 'wb') as f:
    pickle.dump(X_test, f)
with open('../data_1s_30hz_real_world/y_test_onehot.pkl', 'wb') as f:
    pickle.dump(y_test_onehot, f)

## 6. Train, validation Spilt

In [21]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_normalized,
    y_train_onehot,
    test_size=0.2,           # 20% for validation
    random_state=42,
    shuffle=True
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)

Train shape: (352, 30, 3) (352, 2)
Validation shape: (89, 30, 3) (89, 2)


In [22]:
# Save training and validation data
with open('../data_1s_30hz_real_world/X_train_normalized.pkl', 'wb') as f:
    pickle.dump(X_train, f)

with open('../data_1s_30hz_real_world/X_val_normalized.pkl', 'wb') as f:
    pickle.dump(X_val, f)

with open('../data_1s_30hz_real_world/y_train_onehot.pkl', 'wb') as f:
    pickle.dump(y_train, f)

with open('../data_1s_30hz_real_world/y_val_onehot.pkl', 'wb') as f:
    pickle.dump(y_val, f)